In [1]:
import pandas as pd
import numpy as np

# ============================================================
# 1) CARGA DE MAESTROS DESDE EXCEL
# ============================================================

locales = pd.read_excel("locales.xlsx")
franquiciados = pd.read_excel("franquiciados.xlsx")
productos = pd.read_excel("artículos.xlsx")
cod_postales = pd.read_excel("codigos_postales.xlsx")

# Ajuste de nombres
cod_postales = cod_postales.rename(columns={"Código postal": "Cód Postal"})
locales["Fecha Apertura"] = pd.to_datetime(locales["Fecha Apertura"])

# ============================================================
# 2) CALENDARIO 2024–2025
# ============================================================

calendario = pd.DataFrame({
    "Fecha": pd.date_range("2024-01-01", "2025-12-31", freq="D")
})
calendario["mes"] = calendario["Fecha"].dt.month
calendario["dia_sem"] = calendario["Fecha"].dt.weekday
calendario["año"] = calendario["Fecha"].dt.year

# ============================================================
# 3) CATEGORÍA SIMPLIFICADA DE PRODUCTO
# ============================================================

def categoria_producto(row):
    nombre = str(row["producto"]).lower()
    if "jarra" in nombre: return "Jarra"
    if "doble" in nombre: return "Doble"
    if "tercio" in nombre: return "Tercio"
    if "botellín" in nombre or "botellin" in nombre: return "Botellín"
    if "barril" in nombre: return "Barril"
    if row["familia"] == "Bebida" and row["categoria"] == "Refresco": return "Refresco"
    return "Otros"

productos["cat_simplificada"] = productos.apply(categoria_producto, axis=1)

# ============================================================
# 4) MULTIPLICADORES ALEATORIOS EXTREMOS
# ============================================================

# --- Tipo de local (cada local tiene su propio multiplicador aleatorio)
def mult_tipo_local(tipo):
    if tipo == "Pub": return np.random.uniform(0.80, 1.00)
    if tipo == "Cervecería": return np.random.uniform(0.90, 1.15)
    if tipo == "Bar": return np.random.uniform(0.80, 1.05)
    return 1.0

locales["mult_tipo_local"] = locales["Tipo Local"].apply(mult_tipo_local)

# --- Franquiciado (cada franquiciado tiene su propio multiplicador aleatorio)
def mult_perfil(perfil):
    if perfil == "Gold": return np.random.uniform(1.10, 1.25)
    if perfil == "Silver": return np.random.uniform(1.00, 1.15)
    return np.random.uniform(0.90, 1.10)

franquiciados["mult_perfil"] = franquiciados["perfil"].apply(mult_perfil)

# --- Mes (cada mes tiene su propio rango aleatorio)
multiplicador_mes = {
    m: np.random.uniform(0.80, 1.40) for m in range(1, 13)
}

# --- Día de la semana (extremo)
multiplicador_dia_semana = {
    0: np.random.uniform(0.50, 0.80),
    1: np.random.uniform(0.60, 0.85),
    2: np.random.uniform(0.70, 1.00),
    3: np.random.uniform(1.10, 1.40),
    4: np.random.uniform(1.30, 1.60),
    5: np.random.uniform(1.40, 1.80),
    6: np.random.uniform(0.90, 1.20)
}

# --- Producto (cada categoría con rango extremo)
multiplicador_producto = {
    "Jarra": np.random.uniform(1.10, 1.40),
    "Doble": np.random.uniform(1.00, 1.30),
    "Tercio": np.random.uniform(0.80, 1.20),
    "Botellín": np.random.uniform(0.70, 1.10),
    "Barril": np.random.uniform(1.20, 1.60),
    "Refresco": np.random.uniform(0.40, 0.80),
    "Otros": np.random.uniform(0.80, 1.20)
}

# --- Base por producto
base_por_producto = {
    "Jarra": 8, "Doble": 6, "Tercio": 5,
    "Botellín": 4, "Barril": 20,
    "Refresco": 3, "Otros": 2
}

# --- Ubicación (habitantes + aleatorio extremo)
habitantes_media = cod_postales["Habitantes"].mean()
cod_postales["mult_cp"] = cod_postales["Habitantes"].apply(
    lambda h: np.random.uniform(0.7, 1.3) * (h / habitantes_media)
)

# ============================================================
# 5) GRID FECHAS × LOCALES × PRODUCTOS
# ============================================================

calendario["key"] = 1
locales["key"] = 1
productos["key"] = 1

grid = calendario.merge(locales, on="key")
grid = grid[grid["Fecha"] >= grid["Fecha Apertura"]]
grid = grid.merge(productos, on="key")
grid = grid.merge(cod_postales[["Cód Postal", "mult_cp"]], on="Cód Postal", how="left")
grid = grid.merge(franquiciados[["id_franquiciado", "mult_perfil"]],
                  left_on="ID_franquiciado", right_on="id_franquiciado", how="left")

# ============================================================
# 6) TENDENCIA + RUIDO EXTREMO
# ============================================================

fecha_min = calendario["Fecha"].min()
grid["dias_desde_inicio"] = (grid["Fecha"] - fecha_min).dt.days
grid["mult_tendencia"] = 1 + (grid["dias_desde_inicio"] * 0.0002)

np.random.seed(42)
grid["ruido"] = np.random.normal(1.0, 0.25, size=len(grid))  # ruido fuerte

# ============================================================
# 7) CÁLCULO FINAL DE CANTIDAD E IMPORTE
# ============================================================

grid["base_producto"] = grid["cat_simplificada"].map(base_por_producto)

grid["cantidad"] = (
    grid["base_producto"]
    * grid["mult_tipo_local"]
    * grid["mult_perfil"]
    * grid["mes"].map(multiplicador_mes)
    * grid["dia_sem"].map(multiplicador_dia_semana)
    * grid["cat_simplificada"].map(multiplicador_producto)
    * grid["mult_cp"]
    * grid["mult_tendencia"]
    * grid["ruido"]
)

grid["cantidad"] = grid["cantidad"].clip(lower=0).round().astype(int)
grid["importe"] = grid["cantidad"] * grid["precio_venta"]

ventas = grid[["Fecha", "ID_tienda", "id_producto", "cantidad", "importe"]]

# ============================================================
# 8) EXPORTAR 4 CSV SEMESTRALES
# ============================================================
ventas["Fecha"] = pd.to_datetime(ventas["Fecha"]).dt.strftime("%Y-%m-%d")

ventas[(ventas["Fecha"] >= "2024-01-01") & (ventas["Fecha"] <= "2024-06-30")] \
    .to_csv("ventas_2024_S1.csv", index=False)

ventas[(ventas["Fecha"] >= "2024-07-01") & (ventas["Fecha"] <= "2024-12-31")] \
    .to_csv("ventas_2024_S2.csv", index=False)

ventas[(ventas["Fecha"] >= "2025-01-01") & (ventas["Fecha"] <= "2025-06-30")] \
    .to_csv("ventas_2025_S1.csv", index=False)

ventas[(ventas["Fecha"] >= "2025-07-01") & (ventas["Fecha"] <= "2025-12-31")] \
    .to_csv("ventas_2025_S2.csv", index=False)
